# 2. Conversión de proformas

**Pregunta:** ¿qué proformas acabarán en venta y qué lo explica?

**Una fila = una proforma.** **Objetivo:** `convertida` (1 si terminó en venta).

**Método:**
1. Se construyen variables que **se conocen el día en que se emite la proforma**: importe, número de líneas, cliente, vendedor…
2. Partición temporal: se entrena con las proformas de 2024–2025 y se prueba con las de 2026.
3. Se comparan una regresión logística (interpretable) y un *gradient boosting* (más flexible) frente a no usar modelo.
4. Se usa el modelo para ordenar las proformas abiertas por importe esperado.

**Métricas:**
- **AUC:** probabilidad de que el modelo puntúe más alto una proforma que se vendió que una que no. 0,5 = azar; 1 = perfecto.
- **Brier:** error cuadrático de las probabilidades. Cuanto más bajo, mejor; se compara con el de predecir siempre la tasa media.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

from comun import *   # conexion al almacen (con, q), CORTE, guardar(), estilo(), mape(), colores

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 1. Datos: una fila por proforma, con sus dimensiones

In [ ]:
emb = q('''
    SELECT e.num_proforma, CAST(f.fecha AS DATE) AS fecha, CAST(fv.fecha AS DATE) AS fecha_validez,
           e.convertida, ep.estado, vd.vendedor_codigo AS vendedor, c.cliente_id, c.nombre AS cliente,
           c.tipo_cliente, c.sector, c.departamento, CAST(c.fecha_alta AS DATE) AS fecha_alta,
           m.codigo_iso AS moneda, e.importe_ofertado_pen
    FROM fact_embudo_proforma e
    JOIN dim_fecha f            ON f.fecha_key = e.fecha_emision_key
    JOIN dim_fecha fv           ON fv.fecha_key = e.fecha_validez_key
    JOIN dim_vendedor vd        ON vd.vendedor_key = e.vendedor_key
    JOIN dim_cliente c          ON c.cliente_key = e.cliente_key
    JOIN dim_moneda m           ON m.moneda_key = e.moneda_key
    JOIN dim_estado_proforma ep ON ep.estado_proforma_key = e.estado_proforma_key
''')
lineas = q('''
    SELECT num_proforma, COUNT(*) AS n_lineas, COUNT(DISTINCT p.familia) AS n_familias
    FROM fact_proformas fp JOIN dim_producto p ON p.producto_key = fp.producto_key
    GROUP BY 1
''')
emb = emb.merge(lineas, on='num_proforma')
for c in ('fecha', 'fecha_validez', 'fecha_alta'):
    emb[c] = pd.to_datetime(emb[c])
emb.head()

## 2. Variables (ingeniería de características)

Solo información disponible al emitir la proforma. **No** se usan `importe_facturado_pen`, `dias_hasta_venta` ni `estado`: son consecuencias de la venta y darían un modelo perfecto pero inútil (fuga de información).

In [ ]:
emb['antiguedad_cliente_dias'] = (emb.fecha - emb.fecha_alta).dt.days.clip(lower=0)
emb['cliente_recurrente'] = (emb.antiguedad_cliente_dias >= 180).astype(int)  # más de 6 meses como cliente
emb['validez_dias'] = (emb.fecha_validez - emb.fecha).dt.days
emb['log_importe'] = np.log1p(emb.importe_ofertado_pen)  # el logaritmo reduce el peso de las proformas enormes
emb['mes'] = emb.fecha.dt.month
emb['tramo_importe'] = pd.cut(emb.importe_ofertado_pen, [0, 1000, 5000, 1e9],
                              labels=['HASTA S/ 1.000', 'S/ 1.000-5.000', 'MAS DE S/ 5.000']).astype(str)

NUMERICAS = ['log_importe', 'n_lineas', 'n_familias', 'antiguedad_cliente_dias', 'cliente_recurrente', 'validez_dias', 'mes']
CATEGORICAS = ['vendedor', 'tipo_cliente', 'sector', 'departamento', 'moneda']

# Las abiertas aún no tienen resultado: no sirven para entrenar
resueltas = emb[emb.estado != 'ABIERTA'].copy()
print(f'{len(resueltas)} proformas resueltas; tasa de conversión {resueltas.convertida.mean():.1%}')

## 3. Análisis descriptivo: ¿qué se asocia a vender?

Antes de modelar, se miran las tasas de conversión por cada factor. Solo se muestran grupos con al menos 30 proformas.

In [ ]:
factores = []
for col, etiqueta in [('tipo_cliente', 'Tipo de cliente'), ('vendedor', 'Vendedor'), ('tramo_importe', 'Importe ofertado'),
                      ('cliente_recurrente', 'Cliente con más de 6 meses'), ('moneda', 'Moneda'), ('sector', 'Sector')]:
    g = resueltas.groupby(col).agg(proformas=('convertida', 'size'), tasa=('convertida', 'mean'))
    g = g[g.proformas >= 30].reset_index().rename(columns={col: 'valor'})
    g['factor'] = etiqueta
    g['valor'] = g.valor.astype(str).replace({'0': 'NO', '1': 'SI'})
    factores.append(g)
factores = pd.concat(factores)[['factor', 'valor', 'proformas', 'tasa']]
factores['tasa_conversion_pct'] = (factores.pop('tasa') * 100).round(1)
factores['diferencia_vs_media_pp'] = (factores.tasa_conversion_pct - resueltas.convertida.mean() * 100).round(1)
guardar(factores, 'conversion_factores')
factores

In [ ]:
media = resueltas.convertida.mean() * 100
fz = factores[factores.factor.isin(['Tipo de cliente', 'Vendedor', 'Importe ofertado'])].copy()
fz['etiqueta'] = fz.factor + ': ' + fz.valor
fz = fz.sort_values('tasa_conversion_pct')
colores = [ROJO if v < -5 else (VERDE if v > 3 else GRIS) for v in fz.diferencia_vs_media_pp]

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.barh(fz.etiqueta, fz.tasa_conversion_pct, color=colores)
ax.axvline(media, color='#333', lw=1, ls='--')
for i, (v, n) in enumerate(zip(fz.tasa_conversion_pct, fz.proformas)):
    ax.text(v + .5, i, f'{v:.0f} %  (n={n})', va='center', fontsize=8)
ax.set_xlim(0, 85)
estilo(ax, f'Tasa de conversión de proformas por factor (media {media:.0f} %)')
ax.grid(axis='x', color='#e5e5e5'); ax.grid(axis='y', visible=False)
guardar_figura(fig, '2_conversion_factores')
plt.show()

## 4. Entrenamiento y evaluación (entrenar con 2024–2025, probar con 2026)

In [ ]:
entren = resueltas[resueltas.fecha < '2026-01-01']
prueba = resueltas[resueltas.fecha >= '2026-01-01']


def preprocesado():
    # Numéricas: se estandarizan. Categóricas: una columna 0/1 por valor (one-hot).
    return ColumnTransformer([('num', StandardScaler(), NUMERICAS),
                              ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAS)])


modelos = {
    'Regresion logistica': make_pipeline(preprocesado(), LogisticRegression(max_iter=3000, C=0.5)),
    'Gradient boosting': make_pipeline(preprocesado(), HistGradientBoostingClassifier(
        max_depth=3, learning_rate=0.05, max_iter=200, random_state=SEMILLA)),
}
X = NUMERICAS + CATEGORICAS
filas = []
for nombre, modelo in modelos.items():
    if nombre == 'Gradient boosting':  # este modelo necesita una matriz densa
        modelo.steps[0][1].sparse_threshold = 0
    modelo.fit(entren[X], entren.convertida)
    p = modelo.predict_proba(prueba[X])[:, 1]
    filas.append({'modelo': nombre, 'auc': round(roc_auc_score(prueba.convertida, p), 3),
                  'brier': round(brier_score_loss(prueba.convertida, p), 4)})
tasa_media = np.full(len(prueba), entren.convertida.mean())
filas.append({'modelo': 'Referencia: tasa media (sin modelo)', 'auc': 0.5,
              'brier': round(brier_score_loss(prueba.convertida, tasa_media), 4)})
metricas = pd.DataFrame(filas).assign(n_entrenamiento=len(entren), n_prueba=len(prueba))
guardar(metricas, 'conversion_metricas')
metricas

**Cómo leerlo:** un AUC de en torno a 0,55 es **poco mejor que el azar**. Con los datos disponibles **no se puede predecir bien qué proforma se venderá**. El *gradient boosting* no mejora: no hay patrones complejos que la regresión logística se esté perdiendo.

**Esto es un resultado, no un fracaso, y hay que contarlo así.** Lo que decide una venta (precio de la competencia, urgencia del cliente, relación personal) no está en el sistema de facturación. La recomendación a la empresa es **registrar el motivo de pérdida** de cada proforma rechazada o vencida. Lo que sí se ve son diferencias claras por grupos (gráfico anterior): personas naturales y algunos vendedores convierten bastante menos.

## 5. Uso práctico: priorizar las proformas abiertas

Aunque el modelo distingue poco, sus probabilidades están bien calibradas en promedio. Multiplicadas por el importe dan el **importe esperado**, que sirve para ordenar el seguimiento comercial.

In [ ]:
final = make_pipeline(preprocesado(), LogisticRegression(max_iter=3000, C=0.5))
final.fit(resueltas[X], resueltas.convertida)

abiertas = emb[emb.estado == 'ABIERTA'].copy()
abiertas['prob_conversion'] = final.predict_proba(abiertas[X])[:, 1].round(3)
abiertas['importe_esperado_pen'] = (abiertas.prob_conversion * abiertas.importe_ofertado_pen).round(2)
abiertas = abiertas.sort_values('importe_esperado_pen', ascending=False)
abiertas['fecha'] = abiertas.fecha.dt.date
abiertas['fecha_validez'] = abiertas.fecha_validez.dt.date
salida = abiertas[['num_proforma', 'fecha', 'fecha_validez', 'cliente_id', 'cliente', 'tipo_cliente', 'vendedor',
                   'importe_ofertado_pen', 'prob_conversion', 'importe_esperado_pen']]
guardar(salida, 'proformas_abiertas_prioridad')
print(f'{len(salida)} proformas abiertas por S/ {salida.importe_ofertado_pen.sum():,.0f}; '
      f'importe esperado S/ {salida.importe_esperado_pen.sum():,.0f}')
salida.head(10)

## Conclusiones (para la memoria)

- Tasa de conversión global en torno al 64 %.
- Factores con más efecto: **tipo de cliente** (las personas naturales convierten unos 19 puntos menos) y **vendedor** (unos 13 puntos entre el mejor y el peor).
- El modelo predictivo apenas supera al azar (AUC ≈ 0,56): la información del sistema de facturación no basta para anticipar una venta concreta. Se propone registrar el motivo de pérdida y volver a entrenar.
- El importe esperado de las proformas abiertas es una estimación útil para el cuadro de mando.